# MammosighTR — Veri Seti Hazırlama & Meme Kırpma (512x512)

Bu notebook, **Eğitim-1 ve Eğitim-2** veri setlerini sıralı indirip, meme sınırlarını kırparak **512x512 JPEG** formatına dönüştürür. 
Bu sayede tek bir çıktı oluşturur. Diğer YOLO eğitim notebook'larında (v8, v9, v10) bu çıktıyı girdi olarak ekleyerek indirme süresi beklemeden doğrudan eğitime başlayabilirsiniz.

**Not:** Bu işlemi Kaggle'da **GPU harcamadan (CPU seçerek)** arka planda (Save Version -> Save & Run All) çalıştırabilirsiniz. Böylece GPU kotanız korunur.

In [ ]:
# Gerekli kütüphaneleri kur
!pip install pydicom openpyxl beautifulsoup4 requests tqdm pandas numpy Pillow opencv-python-headless stream-unzip -q

import os, sys, glob, shutil, zipfile, io, requests, warnings, socket
import numpy as np, pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
from PIL import Image
import pydicom
import cv2
from sklearn.model_selection import train_test_split
from concurrent.futures import ThreadPoolExecutor, as_completed
from stream_unzip import stream_unzip

warnings.filterwarnings("ignore")
socket.setdefaulttimeout(60)

# ── Ortam Otomatik Algılama (Kaggle / Colab / Lokal PC) ──
if os.path.exists('/kaggle'):
 print("💻 Kaggle ortamı algılandı.")
 TEMP_DIR = "/kaggle/temp/MammosighTR_Data"
 EXTRACTED_DIR = "/kaggle/temp/extracted"
 out_zip_path = "/kaggle/working/yolo_dataset.zip"
elif os.path.exists('/content'):
 print("💻 Google Colab ortamı algılandı.")
 try:
  from google.colab import drive
  print("🔑 Google Drive bağlantısı isteniyor...")
  drive.mount('/content/drive')
  out_zip_path = "/content/drive/MyDrive/yolo_dataset.zip"
 except Exception as e:
  print(f" Google Drive bağlanamadı, çıktı yerel kaydedilecek: {e}")
  out_zip_path = "/content/yolo_dataset.zip"
 TEMP_DIR = "/content/MammosighTR_Data"
 EXTRACTED_DIR = "/content/extracted"
else:
 print("💻 Yerel bilgisayar ortamı algılandı.")
 TEMP_DIR = "./temp_mammosight"
 EXTRACTED_DIR = "./temp_extracted"
 out_zip_path = "./yolo_dataset.zip"

# Temizlik
for path in [TEMP_DIR, EXTRACTED_DIR, out_zip_path]:
 if os.path.exists(path):
  try:
   if os.path.isdir(path): shutil.rmtree(path)
   else: os.remove(path)
  except: pass

os.makedirs(TEMP_DIR, exist_ok=True)
os.makedirs(EXTRACTED_DIR, exist_ok=True)

DATASET_URL = "https://acikveri.saglik.gov.tr/Home/DataSetDetail/3"
BASE_URL = "https://acikveri.saglik.gov.tr"

CLASS_NAMES = ["BIRADS0","BIRADS1_2","BIRADS4_5"]
DISPLAY_NAMES = ["BI-RADS 0","BI-RADS 1-2","BI-RADS 4-5"]


In [ ]:
# ── Bilgi.zip (Excel metadata) İndir ve Çıkar ─────────────────
import subprocess, time

print(" Bilgi.zip indiriliyor...")
res = requests.get(DATASET_URL, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
from bs4 import BeautifulSoup
soup = BeautifulSoup(res.text, "html.parser")
bilgi_url = None
for a in soup.find_all("a", href=True):
 href = a["href"]
 text = a.get_text(strip=True)
 if ("Download" in href or "Document" in href) and "Bilgi" in text:
  bilgi_url = BASE_URL + href if href.startswith("/") else href
  break

if not bilgi_url:
 # Fallback: tüm linklerden Bilgi.zip olanı bul
 for a in soup.find_all("a", href=True):
  href = a["href"]
  if "Download" in href or "Document" in href:
   text = a.get_text(strip=True)
   if "Bilgi" in text or "bilgi" in text.lower():
    bilgi_url = BASE_URL + href if href.startswith("/") else href
    break

if bilgi_url:
 bilgi_dest = os.path.join(TEMP_DIR, "Bilgi.zip")
 cmd = ["curl", "--retry", "5", "--retry-delay", "3",
   "-H", "User-Agent: Mozilla/5.0", "-L", "-f",
   "--connect-timeout", "30", "-m", "120",
   "-o", bilgi_dest, bilgi_url]
 result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
 if result.returncode == 0 and os.path.exists(bilgi_dest):
  print("Bilgi.zip indirildi. Çıkarılıyor...")
  with zipfile.ZipFile(bilgi_dest, "r") as zf:
   zf.extractall(EXTRACTED_DIR)
  os.remove(bilgi_dest)
  print("Bilgi.zip çıkarıldı.")
 else:
  print(f"UYARI: Bilgi.zip indirilemedi! Hata: {result.stderr.decode()}")
else:
 print("UYARI: Bilgi.zip linki bulunamadı!")


In [ ]:
# ── Excel dosyalarını bul ve oku ─────────────────────────────────
xlsx_files = glob.glob(os.path.join(EXTRACTED_DIR, "**", "*.xlsx"), recursive=True)
TRAIN1 = [f for f in xlsx_files if "TRAIN1" in f][0] if any("TRAIN1" in f for f in xlsx_files) else ""
TRAIN2 = [f for f in xlsx_files if "TRAIN2" in f][0] if any("TRAIN2" in f for f in xlsx_files) else ""

print(f"Bulunan Excel Dosyaları:\nTRAIN1: {TRAIN1}\nTRAIN2: {TRAIN2}")

def parse_birads(val):
 s = str(val).strip().upper().replace("BI-RADS","").replace("BIRADS","").replace(" ","")
 nums = [int(c) for c in s if c.isdigit()]
 if not nums: return -1
 n = nums[0]
 return 0 if n==0 else (1 if n in [1,2] else (2 if n in [4,5] else -1))

def read_excel_split(xlsx_path):
 df = pd.read_excel(xlsx_path)
 records = []
 birads_col = next((col for col in df.columns if any(x in str(col).upper() for x in ["BIRADS","BI-RADS","KATEGORİ"])), "BIRADS CATEGORY")
 case_col = next((col for col in df.columns if any(x in str(col).upper() for x in ["CASE","HASTA","ID","NO"])), "CASENUMBER")
 source_name = "train1" if "TRAIN1" in xlsx_path else "train2"
 
 for _, row in df.iterrows():
  case_num = row.get(case_col)
  if pd.isna(case_num): continue
  try: case_num = int(case_num)
  except: continue
   
  birads_val = row.get(birads_col)
  label = parse_birads(birads_val)
  if label == -1: continue
   
  for view in ["RCC.dcm", "LCC.dcm", "RMLO.dcm", "LMLO.dcm"]:
   records.append({
    "case": case_num,
    "view": view,
    "label": label,
    "source": source_name
   })
 return records

recs = []
if TRAIN1: recs += read_excel_split(TRAIN1)
if TRAIN2: recs += read_excel_split(TRAIN2)

df_full = pd.DataFrame(recs)
print(f"Toplam Eşleşecek Vaka Sayısı: {len(df_full)}")

In [ ]:
# ── Train-Val-Test Ayrımı ve Sınıf Dengeleme (Oversampling) ──────
train_df, temp_df = train_test_split(df_full, test_size=0.30, stratify=df_full["label"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=42)

def oversample_df(df):
 max_size = df["label"].value_counts().max()
 balanced_list = []
 for label in df["label"].unique():
  cls_df = df[df["label"] == label]
  if len(cls_df) < max_size:
   cls_df = cls_df.sample(max_size, replace=True, random_state=42)
  balanced_list.append(cls_df)
 return pd.concat(balanced_list).sample(frac=1, random_state=42).reset_index(drop=True)

train_df_balanced = oversample_df(train_df)
val_df_balanced = oversample_df(val_df)

print(f"Dengelenmiş Veri Dağılımı:\n Train: {dict(train_df_balanced['label'].value_counts())}\n Val : {dict(val_df_balanced['label'].value_counts())}\n Test : {dict(test_df['label'].value_counts())}")

# Alt klasörleri oluştur

# Hedef haritasını oluştur
case_destinations = {}
for idx, row in train_df_balanced.iterrows():
 key = (int(row["case"]), row["view"].lower())
 case_destinations.setdefault(key, []).append(("train", idx, CLASS_NAMES[int(row["label"])]))
for idx, row in val_df_balanced.iterrows():
 key = (int(row["case"]), row["view"].lower())
 case_destinations.setdefault(key, []).append(("val", idx, CLASS_NAMES[int(row["label"])]))
for idx, row in test_df.iterrows():
 key = (int(row["case"]), row["view"].lower())
 case_destinations.setdefault(key, []).append(("test", idx, CLASS_NAMES[int(row["label"])]))

active_sources = set(df_full["source"].dropna().unique())
print(f"Aktif Veri Kaynakları: {active_sources}")

In [ ]:
# ── Sıralı İndirme ve Ön İşleme (CLAHE + Meme Kırpma + 512x512 JPEG) ────
import subprocess
import time
import random

# CLAHE filtresi (tıbbi görüntülemede standart kontrast artırma)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

print(" Sağlık Bakanlığı Açık Veri Portalı taranıyor...")
res = requests.get(DATASET_URL, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
soup = BeautifulSoup(res.text, 'html.parser')
links = []
for a in soup.find_all('a', href=True):
 href = a['href']
 if 'Download' in href or 'Document' in href:
  text = a.get_text(strip=True).replace(" İndirmek İçin Tıklayınız", "").replace(" click to download", "")
  url = BASE_URL + href if href.startswith("/") else href
  links.append({"name": text, "url": url})

groups = {}
for item in links:
 name = item["name"]
 if "Bilgi" in name: continue
 group_name = name.split(".zip")[0] + ".zip" if ".zip" in name else name
 groups.setdefault(group_name, []).append(item)

def download_item_curl(item, dest_dir):
 name = item["name"]
 url = item["url"]
 dest = os.path.join(dest_dir, name)
 if os.path.exists(dest) and os.path.getsize(dest) > 0:
  return name, True
  
 # Sunucuya yüklenmemek için rastgele 2 ile 4 saniye arası bekletiyoruz
 time.sleep(random.uniform(2.0, 4.0))
 
 cmd = ["curl", "--retry", "5", "--retry-delay", "3", "-H", "User-Agent: Mozilla/5.0", "-L", "-f", "--connect-timeout", "30", "-m", "300", "-o", dest, url]
 try:
  result = subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
  if result.returncode == 0 and os.path.exists(dest) and os.path.getsize(dest) > 0:
   return name, True
  else:
   try:
    if os.path.exists(dest): os.remove(dest)
   except: pass
   return name, False
 except:
  try:
   if os.path.exists(dest): os.remove(dest)
  except: pass
  return name, False

def get_zipped_chunks(g_items, g_name):
 """Split zip dosyalarını sırayla diske indirip, byte akışını stream-unzip'e gönderen jeneratör."""
 for idx, item in enumerate(g_items):
  name = item["name"]
  dest = os.path.join(TEMP_DIR, name)
  
  # Parçayı indir
  success = False
  for attempt in range(1, 6):
   # curl ile indir ve kontrol et
   _, ok = download_item_curl(item, TEMP_DIR)
   if ok:
    success = True
    break
   print(f"\n {name} indirilemedi. Tekrar denenecek (Deneme {attempt}/5)...")
   time.sleep(15 * attempt)
   
  if not success:
   raise RuntimeError(f"Kritik Hata: {name} parçası hiçbir denemede indirilemedi!")
   
  # İndirilen parçanın byte'larını oku ve jeneratörden yield et
  with open(dest, 'rb') as f:
   while True:
    chunk = f.read(262144) # 256KB chunks
    if not chunk:
     break
    yield chunk
    
  # Yield edildikten hemen sonra diski temizlemek için parçayı sil!
  try: os.remove(dest)
  except: pass
  
  # İlerlemeyi ekrana yazdır
  print(f" - [{idx+1}/{len(g_items)}] {name} akışı tamamlandı ve diskten silindi.")

for g_name, g_items in groups.items():
 is_needed = False
 if "EGITIM_1" in g_name and "train1" in active_sources: is_needed = True
 elif "EGITIM_2" in g_name and "train2" in active_sources: is_needed = True
 
 if not is_needed:
  print(f"Atlanıyor: {g_name}")
  continue
  
 print(f"\n--- {g_name} Akış İndirme ve İşleme Başlatıldı ({len(g_items)} parça) ---")
 sorted_items = sorted(g_items, key=lambda x: x["name"])
 
 ok, skip = 0, 0
 try:
  # Çıktı zip dosyasını açıyoruz
  with zipfile.ZipFile(out_zip_path, 'a', compression=zipfile.ZIP_DEFLATED) as out_zip:
   # stream-unzip kütüphanesiyle split zip parçalarını indirme akışından canlı okuyoruz
   zipped_generator = get_zipped_chunks(sorted_items, g_name)
   
   for file_name_bytes, file_size, unzipped_chunks in stream_unzip(zipped_generator):
    file_name = file_name_bytes.decode('utf-8')
    if not file_name.lower().endswith(".dcm"):
     # İstenmeyen dosyaların chunk'larını tüketip geç
     for _ in unzipped_chunks: pass
     continue
     
    parts = file_name.strip("/").split("/")
    if len(parts) < 2:
     for _ in unzipped_chunks: pass
     continue
     
    casenumber_str = parts[-2]
    filename = os.path.basename(file_name)
    try: casenumber = int(casenumber_str)
    except:
     for _ in unzipped_chunks: pass
     continue
     
    dest_info = case_destinations.get((casenumber, filename.lower()))
    if not dest_info:
     for _ in unzipped_chunks: pass
     continue
     
    # DICOM dosyasını hafızaya çek
    dcm_data = b"".join(unzipped_chunks)
    
    try:
     ds = pydicom.dcmread(io.BytesIO(dcm_data))
     arr = ds.pixel_array.astype(np.float32)
     if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
      arr = arr.max() - arr
     max_val = arr.max()
     if max_val > 0:
      arr_norm = (arr / max_val * 255).astype(np.uint8)
     else:
      arr_norm = arr.astype(np.uint8)
     
     # CLAHE kontrast artırma
     arr_norm = clahe.apply(arr_norm)
     
     # Meme Kırpma (siyah arka planı silme)
     mask = arr_norm > 10
     row_any = np.any(mask, axis=1)
     col_any = np.any(mask, axis=0)
     rows = np.where(row_any)[0]
     cols = np.where(col_any)[0]
     if len(rows) > 0 and len(cols) > 0:
      ymin, ymax = rows[0], rows[-1]
      xmin, xmax = cols[0], cols[-1]
      if ymax > ymin and xmax > xmin:
       arr_norm = arr_norm[ymin:ymax, xmin:xmax]
       
     img = Image.fromarray(arr_norm).convert("RGB").resize((512,512))
     for split, idx, label_name in dest_info:
      buffer = io.BytesIO()
      img.save(buffer, format="JPEG", quality=90)
      arcname = f"{split}/{label_name}/{split}_{idx:06d}.jpg"
      out_zip.writestr(arcname, buffer.getvalue())
      ok += 1
    except:
     skip += 1
 except Exception as e:
  print(f" Hata oluştu: {str(e)}")
  raise e
  
 print(f"Bitti: {g_name} | {ok} JPEG doğrudan zipe yazıldı | {skip} hata. Disk temizlendi.")

In [ ]:
# ── Doğrulama ─────────────────────────────────
print("\n--- Veri Seti İstatistikleri ---")
if os.path.exists(out_zip_path):
 with zipfile.ZipFile(out_zip_path, 'r') as ref:
  namelist = ref.namelist()
  for split in ["train", "val", "test"]:
   total = 0
   print(f"{split.upper()} Klasörü:")
   for cls in CLASS_NAMES:
    prefix = f"{split}/{cls}/"
    cnt = sum(1 for name in namelist if name.startswith(prefix) and name.endswith(".jpg"))
    total += cnt
    print(f" - {cls}: {cnt} resim")
   print(f" Toplam: {total} resim\n")
 print(" yolo_dataset.zip başarıyla doğrulandı! (/kaggle/working/yolo_dataset.zip)")
else:
 print("❌ Hata: /kaggle/working/yolo_dataset.zip dosyası bulunamadı!")